# Product Hunt Launch Success Analytics: Identifying Pre-Launch Factors That Influence Launch-Day Traction

**University Business Analytics — Individual Case Study**

---

## 1. Project Introduction

### 1.1 Background: Product Hunt

Product Hunt is the primary launchpad for new tech products — web apps, SaaS tools, AI software, and mobile tools. Every day at 00:01 PST (midnight Pacific), Product Hunt resets its daily leaderboard. Makers submit their products, and over the next 24 hours the community votes, comments, and reviews them. Products that rank near the top of the daily leaderboard receive prime homepage placement, significant industry exposure, and a credibility boost.

### 1.2 Why Launch Success Matters

For an early-stage startup or solo developer, a successful Product Hunt launch day can be transformative:

- **Immediate Traffic & Users**: Top-5 finishes often bring thousands of visitors, sign-ups, and early paying customers within 48 hours.
- **Investor Attention**: Angel investors and VCs regularly scan Product Hunt's daily top products to discover high-traction teams early.
- **Social Proof & Press**: "Product of the Day" badges build instant credibility and frequently lead to coverage in tech newsletters and blogs.

A poor launch, by contrast, means wasted preparation time and missed momentum.

### 1.3 Business Problem

Product Hunt launches show enormous variance in outcomes — some tools earn thousands of upvotes, others barely reach ten. Founders typically rely on guesswork for critical pre-launch decisions:

1. **Timing**: What day of the week or hour of the day is best?
2. **Content & Copy**: How detailed should the description be? Does adding a video help?
3. **Network**: How much does team size or a well-connected "Hunter" boost votes?
4. **Category**: Does the product category affect competition on the leaderboard?

This project uses data analytics to evaluate which pre-launch choices are **associated with** higher traction, so that founders and growth marketers have a data-driven launch checklist.

### 1.4 Project Objectives

| # | Objective |
|---|-----------|
| 1 | Load, profile, and clean a 10,000-record Product Hunt launch dataset. |
| 2 | Engineer practical pre-launch features (timing windows, social reach, media metrics, text length features). |
| 3 | Define a balanced binary target (`SUCCESS`) representing high-traction launches. |
| 4 | Conduct EDA to identify associations between pre-launch factors and success. |
| 5 | Build and evaluate two classification models: Logistic Regression (interpretable baseline) and XGBoost (non-linear). |
| 6 | Interpret feature importance and SHAP values to extract actionable business insights. |

### 1.5 Scope & Limitations

This analysis focuses strictly on **pre-launch variables** — factors that founders can control or measure *before* pushing the launch button. Post-launch metrics (comment velocity, post-launch conversion, final rank) are excluded as predictors so the findings can serve as a practical pre-launch checklist.

**Limitation**: Upvote counts are a popularity/visibility proxy, not a direct business outcome (revenue, retention, funding). This is acknowledged throughout the analysis.

## 2. Dataset Description

### 2.1 Data Collection Methodology

The analytical dataset was constructed through a two-stage process:

**Stage 1 — Web Scraping (Real Product Hunt Data):**
- Public Product Hunt daily leaderboard pages and individual product launch pages were scraped using browser automation (Playwright) and the Apify actor platform.
- Scrapers captured: product name, tagline, URL, upvote count, comment count, categories/topics, product description, launch timestamp, media assets (screenshots and videos), maker/team member lists, and Hunter information.
- All data was collected from **publicly accessible** Product Hunt pages without authentication or rate-limit violations.

**Stage 2 — Synthetic Data Augmentation (Gretel AI):**
- The real web-scraped sample was used to train a **CTGAN** (Conditional Tabular GAN) synthesizer via the **Gretel AI** platform (approved by faculty for this coursework).
- The synthesizer learned the joint distribution of the real data — preserving column correlations, ranges, and statistical properties.
- 10,000 synthetic records were generated, expanding the dataset to a size suitable for statistical analysis and machine learning.

### 2.2 Dataset Characteristics

| Property | Value |
|----------|-------|
| Total records | 10,000 |
| Total columns | 17 |
| Target variable | `SUCCESS` (binary: 0 = below median, 1 = at/above median upvotes) |
| Source | Real PH web-scraped data → Gretel AI (CTGAN) augmentation |
| Date coverage | May–July 2026 daily leaderboard data |
| PII status | Fully anonymized (Gretel AI removes all personal identifiers) |

### 2.3 Column Dictionary

| Column | Type | Description | Pre-launch? |
|--------|------|-------------|------------|
| `rank` | int | Leaderboard rank | No (post-launch outcome) |
| `votesCount` | int | Total upvotes | No (used for target) |
| `commentsCount` | int | Total comments | No (post-launch engagement) |
| `launch_hour` | int | Hour of launch (0–23) | Yes |
| `weekday` | int | Day of week (0=Mon, 6=Sun) | Yes |
| `month` | int | Month of launch (1–12) | Yes |
| `is_featured` | bool | Whether product was featured | No (post-launch editorial) |
| `description_length` | int | Character count of description | Yes |
| `tagline_length` | int | Character count of tagline | Yes |
| `topic_count` | int | Number of topics/categories | Yes |
| `maker_count` | int | Number of listed makers | Yes |
| `maker_total_followers` | int | Combined maker follower count | Yes |
| `hunter_followers_count` | int | Hunter's follower count | Yes |
| `media_count` | int | Total media assets (screenshots + video) | Yes |
| `has_video` | bool | Whether a video walkthrough exists | Yes |
| `self_launched` | bool | Whether maker launched their own product | Yes |
| `primary_topic` | str | Primary product category | Yes |

> **Note**: `rank`, `votesCount`, `commentsCount`, and `is_featured` are **post-launch variables** and will be excluded from the predictor set. `votesCount` is used **only** to construct the target variable `SUCCESS`.

## 3. Import Libraries

We use standard Python libraries for data manipulation, visualization, and machine learning.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import warnings

# Scikit-learn for preprocessing and baseline modeling
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

# XGBoost for non-linear modeling
from xgboost import XGBClassifier

# SHAP for model explainability
import shap

# Configure plot aesthetics
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

warnings.filterwarnings('ignore')

print("Libraries imported successfully.")

## 4. Data Loading

We load the analytical dataset from the `data/raw/` directory.

In [ ]:
# Load the 10,000-record Product Hunt analytical dataset
DATASET_PATH = 'data/raw/synthetic_producthunt_10000.csv'
df = pd.read_csv(DATASET_PATH)

# Check dataset dimensions
print(f"Dataset Shape: {df.shape[0]:,} rows, {df.shape[1]} columns")
print(f"\nColumn Headers: {df.columns.tolist()}")

### Initial Observations

1. **Dimensions**: 10,000 rows and 17 columns — ample sample size for statistical work and machine learning.
2. **Column name issue**: The first column `'    rank'` has extra leading whitespace. We will clean this in Section 5.
3. **Column types**: The dataset includes integer counts (votes, comments, followers, media), boolean flags (featured, video, self-launched), and one categorical column (`primary_topic`).

## 5. Data Quality Assessment

Before modeling, we systematically assess data quality across four dimensions: structure, missing values, duplicates, and distributions.

### 5.1 Data Structure & Types

We inspect column datatypes and non-null counts to understand the dataset's structural integrity.

In [ ]:
# Inspect column datatypes and non-null counts
print("Column Data Types and Non-Null Counts:")
print(df.dtypes)
print(f"\nMemory usage: {df.memory_usage(deep=True).sum() / 1024:.1f} KB")

**Interpretation**: The dataset is lightweight (~1 MB). All 17 columns are populated without structural issues. Integer columns (`rank`, `votesCount`, `commentsCount`, various counts) are stored as `int64`, boolean flags as `bool`, and `primary_topic` as `object` (string).

### 5.2 Summary Statistics

We examine the distribution of each numerical variable to identify skewness, outliers, and unreasonable values.

In [ ]:
# Get summary statistics for all numerical columns
df.describe().T

**Interpretation of summary statistics**:

- **Upvotes (`votesCount`)**: Mean = 123.6, Median = 70.0, Max = 2,661. The mean exceeds the median substantially, indicating a **right-skewed** distribution. A small number of viral launches dominate total vote volume. This power-law pattern is expected for ranking systems like Product Hunt.

- **Social reach**: `hunter_followers_count` has a median of 1,894 but a max of 170,171 — a massive range. At least 25% of products have 0 hunter followers (median of `maker_total_followers` = 149.5, max = 49,566).

- **Text lengths**: `description_length` ranges from 48 to 500 characters. `tagline_length` ranges from 17 to 60 characters. Both are within reasonable bounds.

- **Timing**: `launch_hour` spans 0–23 (full 24-hour range). `weekday` spans 0–6 (all days represented). `month` spans 1–12.

### 5.3 Missing Values and Duplicate Rows

We check for data gaps and redundant records that could bias the analysis.

In [ ]:
# Check missing value counts
missing = df.isnull().sum()
print("Missing values per column:")
print(missing[missing > 0] if missing.sum() > 0 else "No missing values found.")

# Check for duplicate rows
duplicates = df.duplicated().sum()
print(f"\nTotal duplicate rows: {duplicates}")

**Interpretation**: 0 missing values and 0 duplicate rows. The synthetic data pipeline produced a clean dataset with no imputation or deduplication needed.

### 5.4 Unique Value Profiles

We examine the cardinality of each column to understand category counts and identify potential feature engineering opportunities.

In [ ]:
# Inspect data types, unique value counts, and sample values
profile = pd.DataFrame({
    'Data Type': df.dtypes,
    'Unique Values': df.nunique(),
    'Sample Value': df.iloc[0]
})
profile

**Interpretation**:

- `primary_topic`: 71 unique categories (e.g., Productivity, Mac, iOS, Developer Tools) — useful for one-hot encoding.
- `launch_hour`: 24 unique values (0–23) — represents every possible launch hour.
- `weekday`: 7 unique values (0=Monday through 6=Sunday) — full weekly coverage.
- `is_featured`, `has_video`, `self_launched`: Boolean flags with binary cardinality.

## 6. Data Preparation

We now clean the dataset to prepare it for modeling. The key operations are: fixing column name whitespace, trimming string values, casting types, and handling outlier strategy decisions.

In [ ]:
# Step 1: Strip extra spaces from column names (fixes '    rank' → 'rank')
df.columns = df.columns.str.strip()
print("Cleaned column names:", df.columns.tolist())

In [ ]:
# Step 2: Strip whitespace from string values in categorical columns
df['primary_topic'] = df['primary_topic'].astype(str).str.strip()

# Step 3: Cast integer count columns to proper int type
count_cols = ['votesCount', 'commentsCount', 'hunter_followers_count']
for col in count_cols:
    df[col] = df[col].astype(int)

# Step 4: Ensure boolean columns are proper bool type
bool_cols = ['is_featured', 'has_video', 'self_launched']
for col in bool_cols:
    df[col] = df[col].astype(bool)

print("Updated column data types:")
print(df.dtypes)

### Outlier Strategy

In standard datasets, extreme values are sometimes dropped as outliers. However, on Product Hunt, high upvote counts and large follower counts represent **genuine viral launches**, not data errors.

- **Power-law dynamics**: Product Hunt engagement follows a power-law distribution — the top 5% of launches receive the majority of votes.
- **Our approach**: Rather than deleting extreme values, we use **log-transformations** and **median-based splits** to handle skewness safely while preserving all real data.

This is documented as a deliberate methodology choice, not an omission.

## 7. Feature Engineering

We derive practical business features from the raw columns and define our classification target. Only **pre-launch** features are engineered — post-launch variables are excluded from predictors.

### 7.1 Engineered Features

We create features that capture business logic not directly available in the raw columns:

| Feature | Formula | Business Rationale |
|---------|---------|--------------------|
| `is_weekend` | 1 if weekday ∈ {5,6} else 0 | Tests if weekend launches face less competition |
| `launch_window` | Bin hours into 3 windows | Captures PH daily leaderboard reset cycle |
| `total_social_reach` | maker_followers + hunter_followers | Combined pre-launch audience reach |
| `log_social_reach` | log(1 + total_social_reach) | Normalizes heavy right skew in follower counts |
| `has_makers` | 1 if maker_count > 0 else 0 | Whether the product lists any maker |
| `log_votesCount` | log(1 + votesCount) | Normalizes vote distribution for potential regression use |

In [ ]:
# 1. Weekend Launch Flag (is_weekend)
# weekday coding: 0=Monday ... 5=Saturday, 6=Sunday
df['is_weekend'] = df['weekday'].apply(lambda x: 1 if x in [5, 6] else 0)

# 2. Launch Time Window (launch_window)
# Groups the 24 hours into 3 practical windows relative to the 00:01 PST reset
def get_launch_window(hour):
    if 0 <= hour <= 5:
        return 'PST Reset (00:00-05:59)'
    elif 6 <= hour <= 17:
        return 'Business Hours (06:00-17:59)'
    else:
        return 'Evening Off-Peak (18:00-23:59)'

df['launch_window'] = df['launch_hour'].apply(get_launch_window)

# 3. Total Pre-Launch Social Reach (total_social_reach)
df['total_social_reach'] = df['maker_total_followers'] + df['hunter_followers_count']

# 4. Log-Transformed Social Reach (log_social_reach)
df['log_social_reach'] = np.log1p(df['total_social_reach'])

# 5. Has Makers Flag (has_makers)
df['has_makers'] = df['maker_count'].apply(lambda x: 1 if x > 0 else 0)

# 6. Log-Transformed Votes (log_votesCount) — used for target construction reference only
df['log_votesCount'] = np.log1p(df['votesCount'])

print("Engineered features created.")

### 7.2 Target Variable Definition

We define a binary target variable `SUCCESS` that represents **high-traction launches**:

- **`SUCCESS = 1`**: Product received votes at or above the median (≥ 70 upvotes) — indicating above-average launch-day traction.
- **`SUCCESS = 0`**: Product received votes below the median (< 70 upvotes) — indicating below-average traction.

**Rationale for median split**: The median (70 votes) naturally separates the top 50% from the bottom 50% of launches. This creates a **balanced** target (approximately 50/50 class split), avoiding class imbalance issues while clearly distinguishing above-average from below-average performers.

> `votesCount` is used **only** to construct the target and is **excluded** from the predictor features to prevent target leakage.

In [ ]:
# Define binary target: SUCCESS = 1 if votesCount >= median, else 0
votes_median = df['votesCount'].median()
df['SUCCESS'] = (df['votesCount'] >= votes_median).astype(int)

print(f"Median votesCount (threshold): {votes_median}")
print(f"Target distribution:")
print(df['SUCCESS'].value_counts(normalize=True) * 100)

In [ ]:
# Preview the engineered features alongside the target
engineered_cols = [
    'votesCount', 'log_votesCount', 'SUCCESS', 'is_weekend',
    'launch_window', 'total_social_reach', 'log_social_reach', 'has_makers'
]
df[engineered_cols].head(10)

### Business Rationale for Key Features

- **`launch_hour`**: Critical because Product Hunt resets leaderboards at 00:01 PST. Launching early gives products a full 24-hour window to accumulate votes.
- **`weekday`**: Evaluates weekday vs. weekend competition patterns — weekend days may have fewer competing launches.
- **`launch_window`**: Groups hours into operationally meaningful windows aligned with the PH daily cycle.
- **`total_social_reach` / `log_social_reach`**: Combined pre-launch audience size of makers and Hunter. Larger reach → more initial votes.
- **`maker_count` / `team_size`**: More makers bring more combined social networks to amplify the launch.
- **`description_length` / `tagline_length`**: Measures copy detail — detailed descriptions may improve conversion, while concise taglines may be shareable.
- **`media_count` / `has_video`**: Visual assets — video demos help explain complex software; screenshots show product quality.
- **`topic_count` / `primary_topic`**: Product positioning and category competition density.
- **`self_launched`**: Whether the maker launched their own product (no external Hunter). Self-submitted launches may lack the amplification power of an established Hunter.
- **`hunter_followers_count`**: A well-known Hunter's follower reach can significantly boost initial visibility.

## 8. Final Quality Checks

We run four validation checks to confirm the dataset is clean and analyst-ready.

In [ ]:
# Check 1: Verify no missing values remain
print(f"1. Remaining Missing Values: {df.isnull().sum().sum()}")

# Check 2: Verify final dataset dimensions
print(f"2. Final Dataset Shape: {df.shape[0]:,} rows, {df.shape[1]} columns")

# Check 3: Verify target class balance
print("\n3. SUCCESS Target Class Distribution:")
print(df['SUCCESS'].value_counts(normalize=True) * 100)

# Check 4: Summary statistics for all features
print("\n4. Final Descriptive Statistics:")
df.describe().T[['mean', 'std', 'min', '50%', 'max']]

In [ ]:
# Save the cleaned analytical dataset to the processed directory
PROCESSED_PATH = 'data/processed/producthunt_ml_ready.csv'
df.to_csv(PROCESSED_PATH, index=False)
print(f"Cleaned dataset saved to: {PROCESSED_PATH}")
print(f"Shape: {df.shape}")

### Quality Check Summary

| Check | Result |
|-------|--------|
| Missing values | 0 remaining — all columns complete |
| Dataset shape | 10,000 rows × 24 columns (17 original + 7 engineered) |
| Target balance | 50.4% SUCCESS=1, 49.6% SUCCESS=0 — perfectly balanced, no resampling needed |
| Data integrity | All features have clean data types and reasonable ranges |

The dataset is clean, properly formatted, and ready for **Exploratory Data Analysis** (next section) and **Machine Learning Modeling**.